In [ ]:
## Data Preparation

# This notebook constructs the modelling datasets used in the subsequent experiments.
# It identifies the Russian and non-Russian software domains, restricts the data to single-label
# CWE records, and defines the shared 30-class label space.

In [2]:
import re
import unicodedata

import pandas as pd

from src.config import (
    LABEL_COL,
    RU_MODELLING_FILE,
    NONRU_MODELLING_FILE,
    ensure_project_directories,
)

from src.data import (
    load_raw_fstec,
    load_russian_vendors,
)

ensure_project_directories()

In [ ]:
## Load the Raw FSTEC Dataset

# The original FSTEC BDU dataset is loaded and its expected record count is verified
# to ensure that the analysis starts from the intended source data.

In [3]:
df = load_raw_fstec()

print("Raw FSTEC:", df.shape)

assert len(df) == 89987

Raw FSTEC: (89987, 32)


In [ ]:
## Normalise Vendor and Product Names

# Vendor and product strings are normalised to reduce superficial formatting differences
# before applying the Russian-domain identification rules.

In [4]:
def normalize_name(value):

    if pd.isna(value):
        return ""

    value = unicodedata.normalize(
        "NFKC",
        str(value)
    )

    value = value.lower()
    value = value.replace("ё", "е")

    value = (
        value
        .replace("«", '"')
        .replace("»", '"')
        .replace("“", '"')
        .replace("”", '"')
    )

    value = re.sub(
        r"\s+",
        " ",
        value
    )

    return value.strip()


df["_vendor_norm"] = (
    df["Вендор ПО"]
    .apply(normalize_name)
)

df["_product_norm"] = (
    df["Название ПО"]
    .apply(normalize_name)
)

In [ ]:
## Identify Russian Software Register Records

# Records containing an explicit reference to the Unified Register of Russian Software
# are identified as one source of evidence for Russian-domain membership.

In [5]:
registry_mask = (
    df["_product_norm"]
    .str.contains(
        "запись в едином реестре российских программ",
        regex=False,
        na=False
    )
)

print(
    "Records with Russian Software Register mark:",
    registry_mask.sum()
)

Records with Russian Software Register mark: 21489


In [ ]:
## Match Russian Vendors

# A curated vendor list is used to identify records associated with Russian software vendors.
# Vendor names are normalised before matching to improve consistency.

In [6]:
vendors_lookup = load_russian_vendors()

russian_vendors = {
    normalize_name(value)
    for value
    in vendors_lookup["vendor"].dropna()
}


def contains_russian_vendor(value):

    value = normalize_name(value)

    if not value:
        return False

    return any(
        vendor in value
        for vendor in russian_vendors
    )


vendor_mask = (
    df["Вендор ПО"]
    .apply(contains_russian_vendor)
)

In [ ]:
## Construct Russian and Non-Russian Domains

# A record is assigned to the Russian domain if it matches either the Russian Software Register
# criterion or the curated Russian-vendor criterion. All remaining records form the non-Russian comparison domain.

In [7]:
df["_registry_match"] = registry_mask
df["_vendor_match"] = vendor_mask

df["_is_russian"] = (
    df["_registry_match"]
    |
    df["_vendor_match"]
)


ru_domain = (
    df[df["_is_russian"]]
    .copy()
)

nonru_domain = (
    df[~df["_is_russian"]]
    .copy()
)


print("Russian:", len(ru_domain))
print("Non-Russian:", len(nonru_domain))

assert (
    len(ru_domain)
    + len(nonru_domain)
    == len(df)
)

Russian: 22408
Non-Russian: 67579


In [ ]:
## Restrict the Data to Single-Label CWE Records

# CWE identifiers are extracted from each record and only observations associated with exactly one CWE are retained.
# This produces an unambiguous target label for supervised classification.

In [8]:
CWE_PATTERN = re.compile(
    r"CWE-\d+",
    flags=re.IGNORECASE
)


def extract_cwes(value):

    if pd.isna(value):
        return []

    return sorted({
        match.upper()
        for match
        in CWE_PATTERN.findall(str(value))
    })


ru_domain["_cwe_list"] = (
    ru_domain[LABEL_COL]
    .apply(extract_cwes)
)

nonru_domain["_cwe_list"] = (
    nonru_domain[LABEL_COL]
    .apply(extract_cwes)
)


ru_single = (
    ru_domain[
        ru_domain["_cwe_list"].str.len() == 1
    ]
    .copy()
)

nonru_single = (
    nonru_domain[
        nonru_domain["_cwe_list"].str.len() == 1
    ]
    .copy()
)


ru_single["_cwe_target"] = (
    ru_single["_cwe_list"].str[0]
)

nonru_single["_cwe_target"] = (
    nonru_single["_cwe_list"].str[0]
)


print("RU single-label:", len(ru_single))
print("Non-RU single-label:", len(nonru_single))

RU single-label: 19820
Non-RU single-label: 58079


In [ ]:
## Define the Shared CWE Label Space

# Only CWE classes represented by at least 100 records in both domains are retained.
# This produces a common 30-class prediction task that supports meaningful within-domain and cross-domain comparison.

In [9]:
ru_counts = (
    ru_single["_cwe_target"]
    .value_counts()
)

nonru_counts = (
    nonru_single["_cwe_target"]
    .value_counts()
)


ru_supported = set(
    ru_counts[
        ru_counts >= 100
    ].index
)

nonru_supported = set(
    nonru_counts[
        nonru_counts >= 100
    ].index
)


shared_cwes = sorted(
    ru_supported
    &
    nonru_supported
)


print(
    "Shared CWE classes:",
    len(shared_cwes)
)

print(shared_cwes)

assert len(shared_cwes) == 30

Shared CWE classes: 30
['CWE-119', 'CWE-120', 'CWE-121', 'CWE-122', 'CWE-125', 'CWE-190', 'CWE-20', 'CWE-200', 'CWE-22', 'CWE-264', 'CWE-269', 'CWE-284', 'CWE-287', 'CWE-362', 'CWE-399', 'CWE-400', 'CWE-401', 'CWE-404', 'CWE-415', 'CWE-416', 'CWE-476', 'CWE-665', 'CWE-770', 'CWE-78', 'CWE-787', 'CWE-79', 'CWE-835', 'CWE-843', 'CWE-89', 'CWE-94']


In [ ]:
## Construct the Modelling Datasets

# The Russian and non-Russian single-label datasets are restricted to the shared 30 CWE classes selected above.

In [10]:
ru_model = (
    ru_single[
        ru_single["_cwe_target"]
        .isin(shared_cwes)
    ]
    .copy()
)

nonru_model = (
    nonru_single[
        nonru_single["_cwe_target"]
        .isin(shared_cwes)
    ]
    .copy()
)


print(
    "Russian modelling dataset:",
    len(ru_model)
)

print(
    "Non-Russian modelling dataset:",
    len(nonru_model)
)

Russian modelling dataset: 14451
Non-Russian modelling dataset: 41663


In [ ]:
## Export the Modelling Datasets

# Temporary helper columns used during domain identification and class selection
# are removed before the final Russian and non-Russian modelling datasets are saved.

In [11]:
helper_columns = [
    "_vendor_norm",
    "_product_norm",
    "_registry_match",
    "_vendor_match",
    "_is_russian",
    "_cwe_list",
    "_cwe_target",
]


ru_export = ru_model.drop(
    columns=helper_columns,
    errors="ignore"
)

nonru_export = nonru_model.drop(
    columns=helper_columns,
    errors="ignore"
)


ru_export.to_excel(
    RU_MODELLING_FILE,
    index=False
)

nonru_export.to_excel(
    NONRU_MODELLING_FILE,
    index=False
)


print("Modelling datasets saved.")

Modelling datasets saved.
